# SAC — Soft Actor-Critic (2018)

_Papers · Reinforcement Learning_

**Train a stochastic policy that maximizes reward PLUS entropy, with twin Q-critics, so off-policy continuous control becomes stable and sample-efficient.**

---

This notebook builds SAC up one step at a time — first the maximum-entropy objective on a worked example, then a squashed-Gaussian policy and twin Q-critics, then the full training loop with an entropy-off ablation. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch (+ gymnasium for a real env)

We build SAC in four steps: (1) sanity-check the max-entropy objective on the lesson's 3-action worked example, (2) define the squashed-Gaussian policy and twin Q-critics, (3) write the SAC training loop, (4) run it with entropy on and an entropy-off ablation.

### Step 1 — Check the max-entropy objective on a worked example

SAC maximizes reward *plus* entropy: `E[Q] + alpha * H(pi)` (Eq. 1). Its optimal policy is `pi ∝ exp(Q/alpha)` (Eq. 4), a softmax over Q-values. On the lesson's 3-action example `Q = [2, 1, 0]` with `alpha = 1`, we compute the soft policy, its expected Q, its entropy, and the full objective — and confirm it beats the greedy policy, which earns more raw reward but zero entropy.

In [ ]:
# In Colab, for a real continuous-control env first run:  !pip install gymnasium
# (torch is preinstalled in Colab; gymnasium is not.) The toy task below needs only torch.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

torch.manual_seed(0)

# Sanity-check the lesson's worked example: Q = [2,1,0], alpha = 1.
Q = torch.tensor([2.0, 1.0, 0.0])
alpha = 1.0
pi = torch.softmax(Q / alpha, dim=0)              # pi ∝ exp(Q/alpha)  (Eq. 4 target)
expected_Q = (pi * Q).sum()
entropy = -(pi * torch.log(pi)).sum()             # H(pi) in nats
sac_obj = expected_Q + alpha * entropy            # Eq. 1, one step
greedy_obj = Q.max()                              # greedy: E[Q]=2, H=0

print("soft policy        :", [round(p, 3) for p in pi.tolist()])   # [0.665, 0.245, 0.09]
print("E[Q] (soft)        :", round(expected_Q.item(), 3))          # 1.575
print("entropy H (nats)   :", round(entropy.item(), 3))             # 0.834
print("SAC objective soft :", round(sac_obj.item(), 3))             # 2.409
print("SAC objective greedy:", round(greedy_obj.item(), 3))         # 2.000  -> soft wins
assert sac_obj > greedy_obj


### Step 2 — Define the toy task, the policy, and the twin critics

The toy task is a one-step continuous-control problem: a state `s ~ U(-1,1)`, with reward `-(a - s)^2` so the best action matches the state. The policy is a squashed-Gaussian (Eqs. 11-12): sample from a Gaussian via the reparameterization trick, squash with `tanh`, and apply the `tanh` change-of-variables log-prob correction (Appendix C). The twin critics are two independent `Q(s, a)` networks whose minimum we'll use to curb overestimation.

In [ ]:
# A tiny continuous-control toy task.
# State s ~ U(-1,1). Reward is highest when the (squashed, in [-1,1]) action a == s,
# i.e. r = -(a - s)^2. A good policy must read s and act accordingly -> a real
# state-conditioned continuous-control problem, but trivially fast.
def sample_states(n):
    return torch.rand(n, 1) * 2 - 1

def reward(s, a):
    return -(a - s).pow(2)                         # in (-inf, 0], max 0 at a == s

# Squashed-Gaussian policy (Eqs. 11-12 + tanh correction, Appendix C).
class Policy(nn.Module):
    def __init__(self, s_dim=1, a_dim=1, h=64):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(s_dim, h), nn.ReLU(),
                                  nn.Linear(h, h), nn.ReLU())
        self.mu      = nn.Linear(h, a_dim)
        self.log_std = nn.Linear(h, a_dim)

    def sample(self, s):
        h = self.body(s)
        mu = self.mu(h)
        log_std = self.log_std(h).clamp(-5, 2)
        std = log_std.exp()
        dist = Normal(mu, std)
        u = dist.rsample()                          # reparameterized: a = f(eps; s)
        a = torch.tanh(u)                           # squash into [-1, 1]
        # tanh change-of-variables correction (Appendix C):
        logp = dist.log_prob(u) - torch.log(1 - a.pow(2) + 1e-6)
        return a, logp.sum(-1, keepdim=True)

# Twin Q-critics Q(s,a) (Track B: nn.Linear primitives).
class Qnet(nn.Module):
    def __init__(self, s_dim=1, a_dim=1, h=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(s_dim + a_dim, h), nn.ReLU(),
                                 nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 1))
    def forward(self, s, a):
        return self.net(torch.cat([s, a], dim=-1))


### Step 3 — Write the SAC training loop

Each step: sample states, roll out the policy, and form the soft target. With `gamma = 0` the task is single-step (a contextual bandit), so the target is just the reward. We regress *both* critics to that target (Eqs. 2-3), then update the policy to maximize `min(Q1, Q2) - alpha * logp` (Eq. 4) — the `-alpha * logp` term is the entropy bonus. Finally we Polyak-average the target critics and periodically log a greedy-eval reward and the policy's action spread.

In [ ]:
def train(alpha=0.2, steps=1500, gamma=0.0, tau=0.01, batch=256):
    # gamma=0: this toy task is single-step (a contextual bandit), so the soft
    # value target reduces to r + (-alpha*logp) -- entropy still drives everything.
    torch.manual_seed(0)
    pi = Policy()
    q1, q2 = Qnet(), Qnet()
    q1t, q2t = Qnet(), Qnet()
    q1t.load_state_dict(q1.state_dict())
    q2t.load_state_dict(q2.state_dict())
    opt_pi = torch.optim.Adam(pi.parameters(), lr=3e-3)
    opt_q = torch.optim.Adam(list(q1.parameters()) + list(q2.parameters()), lr=3e-3)

    hist = []   # (step, avg reward, avg action-std) -- our exploration proxy
    for t in range(steps):
        s = sample_states(batch)
        with torch.no_grad():
            a, _ = pi.sample(s)
            r = reward(s, a)
            target = r                              # single-step: y = r (Eq. 2 with gamma=0)

        # Critic update: regress BOTH critics to the soft target (Eqs. 2-3).
        loss_q = F.mse_loss(q1(s, a), target) + F.mse_loss(q2(s, a), target)
        opt_q.zero_grad()
        loss_q.backward()
        opt_q.step()

        # Policy update: maximize min(Q1,Q2) - alpha*logp  (Eq. 4).
        a_new, logp = pi.sample(s)
        q_min = torch.min(q1(s, a_new), q2(s, a_new))
        loss_pi = (alpha * logp - q_min).mean()     # entropy term = -alpha*logp
        opt_pi.zero_grad()
        loss_pi.backward()
        opt_pi.step()

        # Polyak soft-update of target critics.
        with torch.no_grad():
            for p, pt in zip(q1.parameters(), q1t.parameters()):
                pt.mul_(1 - tau).add_(tau * p)
            for p, pt in zip(q2.parameters(), q2t.parameters()):
                pt.mul_(1 - tau).add_(tau * p)

        if t % 100 == 0 or t == steps - 1:
            with torch.no_grad():
                se = sample_states(512)
                he = pi.body(se)
                mu = pi.mu(he)
                std = pi.log_std(he).clamp(-5, 2).exp().mean().item()   # action spread
                ae = torch.tanh(mu)
                re = reward(se, ae).mean().item()                       # greedy-eval reward
            hist.append((t, re, std))
            print(f"  step {t:4d}  eval reward (greedy mean): {re:7.4f}   action std: {std:6.4f}")
    return hist


### Step 4 — Train with entropy, then ablate it away

Now we run the loop twice with everything identical except `alpha`. The full SAC run (`alpha = 0.2`) keeps a healthy action spread while reward climbs toward 0. The ablation (`alpha = 0.0`) drops the entropy bonus, so the policy can collapse to near-deterministic and commit prematurely, giving lower or noisier reward. Comparing the two trails isolates the entropy term as the source of SAC's exploration.

In [ ]:
print("\nSAC with entropy (alpha = 0.2):")
sac_hist = train(alpha=0.2)

print("\nABLATION -- entropy OFF (alpha = 0.0), everything else identical:")
abl_hist = train(alpha=0.0)

print("\nWith entropy  (reward, action-std) trail:", [(s, round(r, 3), round(d, 3)) for s, r, d in sac_hist[-4:]])
print("No entropy    (reward, action-std) trail:", [(s, round(r, 3), round(d, 3)) for s, r, d in abl_hist[-4:]])
# With entropy: the policy keeps a healthy action spread while reward climbs toward 0.
# alpha=0 ablation: the action std collapses faster and the policy can prematurely
# commit, giving lower / noisier reward. (Our small run, not the paper's numbers.)


## Visualize the data & results

_Does SAC's maximum-entropy objective (reward + alpha*entropy) keep the policy exploring while reward improves, and does turning the entropy term OFF (alpha = 0, everything else identical) make the policy collapse? We train both on the toy continuous-control task and plot the policy's action spread (exploration) per step._

In [ ]:
# Sketch of how the curves above are produced (full loop in the CODE cell).
# Train SAC with entropy and the alpha=0 ablation on the toy continuous-control task
# with identical policy / twin critics / lr / batches / seed, recording the policy's
# mean action std (exploration) and greedy-eval reward per step.
#
#   sac_hist = train(alpha=0.2)   # green: action spread stays healthy, reward -> ~0
#   abl_hist = train(alpha=0.0)   # red:   action spread collapses, reward plateaus worse
#
# Entropy on  -> policy keeps exploring (Eq. 1's alpha*H term pays for randomness).
# alpha = 0   -> the -alpha*logp term vanishes, the policy collapses to near-deterministic
#               and can commit prematurely: lower, noisier reward.
# (Numbers are from our own toy run; the paper reports MuJoCo results -- Hopper, Walker2d,
#  HalfCheetah, Ant, Humanoid -- and stability across seeds, not these toy values.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The worked objective. At a state with 3 actions the critic gives $Q = [2.0, 1.0, 0.0]$ and
            $\alpha = 1$. Compute the soft policy $\pi \propto \exp(Q/\alpha)$, its expected Q, its entropy,
            and the SAC objective $\mathbb{E}[Q] + \alpha\mathcal{H}$ — and compare it against the greedy
            policy $[1,0,0]$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Soft policy: $\exp([2,1,0]) = [7.389, 2.718, 1.000]$, sum $= 11.107$, so $\pi = [0.665, 0.245, 0.090]$. — _Eq. 4's target is $\pi \propto \exp(Q/\alpha)$; normalizing the exponentiated Q-values gives the soft policy._
- Expected Q: $0.665(2) + 0.245(1) + 0.090(0) = 1.575$; entropy $\mathcal{H} = -\sum\pi\ln\pi = 0.834$ nats. — _Expected Q is the reward part; entropy measures the spread — both feed the Eq. 1 objective._
- SAC objective: $1.575 + 1\times 0.834 = 2.409$. Greedy: $\mathbb{E}[Q] = 2.0$, $\mathcal{H} = 0$, objective $= 2.0$. — _Greedy earns more raw reward (2.0 vs 1.575) but zero entropy; the soft policy's entropy bonus more than compensates._

**Answer:** The soft policy scores $2.409 \gt 2.0$, so it beats the greedy policy on the SAC objective
                 even though it earns less raw reward. This is exactly why SAC keeps a stochastic policy: at $\alpha = 1$,
                 staying spread out is worth more than the small reward sacrificed. The notebook recomputes
                 $\pi = [0.665, 0.245, 0.090]$, $\mathbb{E}[Q] = 1.575$, $\mathcal{H} = 0.834$, objective $= 2.409$.

</details>

**Problem 2.** The ablation. Your SAC agent learns a good, exploratory policy on the toy task. Set the entropy
            temperature $\alpha = 0$ &mdash; keeping the twin critics, networks, replay buffer, and seed identical
            &mdash; and retrain. What happens to the policy's action spread and to the reward, and what does that
            isolate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change only $\alpha$: set it to $0$, so the value target drops the $-\alpha\log\pi$ term and the policy loss becomes just $-\min(Q_1,Q_2)$. — _An honest ablation changes exactly one thing — the entropy term — so any difference is attributable to it._
- Retrain and watch the policy's action standard deviation collapse toward $0$ and exploration die; reward becomes lower and/or more seed-sensitive. — _With no entropy bonus the policy is rewarded only for exploiting, so it commits early to a (possibly suboptimal) action and stops exploring._
- Conclude the entropy term, not the twin critics or the network, is what drives SAC's sustained exploration. — _Both runs share architecture, critics, and data; only the $\alpha\gt 0$ run keeps exploring, isolating entropy as the cause._

**Answer:** With $\alpha = 0$ the policy's spread collapses and it stops exploring &mdash; reward plateaus
                 lower and becomes more seed-sensitive &mdash; while the full-SAC ($\alpha\gt 0$) run keeps a
                 healthy action spread and reaches higher, steadier reward. Since the only difference is the
                 maximum-entropy term (Eq. 1), this isolates reward-plus-entropy as the source of SAC's exploration
                 and stability. The CODEVIZ panel shows this contrast.

</details>

**Problem 3.** Both DDPG and SAC are off-policy continuous-control methods that learn a Q-critic. Name the two key
            things SAC changes versus DDPG, and say why each improves stability.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- DDPG uses a DETERMINISTIC actor (one action per state) + external exploration noise; SAC uses a STOCHASTIC squashed-Gaussian actor that explores via its own samples and maximizes reward + $\alpha$ entropy. — _The entropy-augmented objective makes the optimal policy $\propto \exp(Q/\alpha)$ — a smooth distribution that cannot collapse to a spike, so the actor–critic pair is far less brittle._
- DDPG (vanilla) uses a SINGLE Q-critic; SAC uses TWIN critics and their minimum. — _Two critics rarely overestimate the same action, so the min is a pessimistic, lower-bias target — curbing the overestimation that destabilizes a single critic._

**Answer:** (1) Stochastic + max-entropy actor instead of DDPG's deterministic actor: exploration is
                 built into the objective (reward + $\alpha$ entropy), so no fragile external noise process and no
                 collapse to a single action. (2) Twin Q-critics + their minimum instead of one critic:
                 curbs value overestimation. Together these are why SAC trains stably across seeds where DDPG is
                 brittle &mdash; the comparison the rl-continuous-control concept lesson develops.

</details>